In [1]:
import requests
import pandas as pd
import time
import numpy as np

def get_binance_data(symbol="BTCUSDT", interval="1h", total_candles=30000):
    url = "https://data-api.binance.vision/api/v3/klines"
    all_data = []
    end_time = None

    while len(all_data) < total_candles:
        params = {"symbol": symbol, "interval": interval, "limit": 1000}
        if end_time is not None:
            params["endTime"] = end_time
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        if not data:
            break
        all_data = data + all_data
        end_time = data[0][0] - 1
        time.sleep(0.1)

    all_data = all_data[-total_candles:]
    columns = ["open_time", "open", "high", "low", "close", "volume",
               "close_time", "quote_asset_volume", "number_of_trades",
               "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"]
    df = pd.DataFrame(all_data, columns=columns)
    df = df[["open_time", "open", "high", "low", "close", "volume"]]
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    for column in ["open", "high", "low", "close", "volume"]:
        df[column] = df[column].astype(float)
    df = df.drop_duplicates("open_time").sort_values("open_time").reset_index(drop=True)
    return df



,open_time,open,high,low,close,volume
0,2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
1,2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2,2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
3,2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
4,2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [ ]:
df = get_binance_data(symbol="BTCUSDT", interval="1h", total_candles=100000)
df.head()

In [ ]:
!pip install -q pandas-ta-classic
import pandas_ta_classic as ta



In [ ]:
df["SMA_10"] = ta.sma(df["close"], length=10)
df["SMA_50"] = ta.sma(df["close"], length=50)
df["EMA_10"] = ta.ema(df["close"], length=10)
df["EMA_50"] = ta.ema(df["close"], length=50)
df["RSI"] = ta.rsi(df["close"], length=14)

macd = ta.macd(df["close"])
df = pd.concat([df, macd], axis=1)

bbands = ta.bbands(df["close"], length=20)
df = pd.concat([df, bbands], axis=1)

df["ATR"] = ta.atr(df["high"], df["low"], df["close"], length=14)
df["return_1h"] = df["close"].pct_change(1)
df["return_5h"] = df["close"].pct_change(5)
df["volatility_10"] = df["return_1h"].rolling(10).std()
df["candle_range"] = (df["high"] - df["low"]) / df["close"]
df["candle_body"] = (df["close"] - df["open"]) / df["open"]



In [ ]:
df = df.dropna().reset_index(drop=True)
df.head()

In [ ]:
FUTURE_WINDOW = 10
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15

df["future_return"] = (df["close"].shift(-FUTURE_WINDOW) - df["close"]) / df["close"]
df = df.dropna(subset=["future_return"]).reset_index(drop=True)
df["label"] = (df["future_return"] > 0).astype(int)

train_size = int(len(df) * TRAIN_FRAC)
val_size = int(len(df) * VAL_FRAC)

feature_cols = [
    "return_1h", "return_5h", "SMA_10", "SMA_50", "EMA_10", "EMA_50",
    "RSI", "MACD_12_26_9", "MACDh_12_26_9", "MACDs_12_26_9",
    "BBP_20_2.0", "ATR", "volatility_10", "candle_range", "candle_body"
]

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size:].copy()



In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
X_val = scaler.transform(val_df[feature_cols])
X_test = scaler.transform(test_df[feature_cols])

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values

print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
WINDOW_SIZE = 60

X_val_context = np.concatenate([X_train[-WINDOW_SIZE + 1:], X_val])
y_val_context = np.concatenate([y_train[-WINDOW_SIZE + 1:], y_val])
X_test_context = np.concatenate([X_val[-WINDOW_SIZE + 1:], X_test])
y_test_context = np.concatenate([y_val[-WINDOW_SIZE + 1:], y_test])



In [ ]:
def create_sequences(features, labels, window_size=60):
    X, y = [], []
    for i in range(window_size - 1, len(features)):
        X.append(features[i - window_size + 1:i + 1])
        y.append(labels[i])
    return np.array(X), np.array(y)



In [ ]:
X_train_seq, y_train_seq = create_sequences(X_train, y_train, WINDOW_SIZE)
X_val_seq, y_val_seq = create_sequences(X_val_context, y_val_context, WINDOW_SIZE)
X_test_seq, y_test_seq = create_sequences(X_test_context, y_test_context, WINDOW_SIZE)

print(X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)

In [ ]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model_lstm = Sequential([
    Input(shape=(WINDOW_SIZE, len(feature_cols))),
    LSTM(32, return_sequences=False),
    Dropout(0.4),
    Dense(16, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)

history = model_lstm.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_pred_prob = model_lstm.predict(X_test_seq, verbose=0).flatten()
val_pred_prob = model_lstm.predict(X_val_seq, verbose=0).flatten()
test_pred = (test_pred_prob > 0.5).astype(int)

print(classification_report(y_test_seq, test_pred))



In [ ]:
cm = confusion_matrix(y_test_seq, test_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["DOWN", "UP"]).plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
def sharpe_ratio(returns, periods_per_year=24*365):
    if returns.std() == 0:
        return 0
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)

def max_drawdown(cumulative):
    peak = np.maximum.accumulate(cumulative)
    return ((cumulative - peak) / peak).min()

def win_rate(returns):
    traded = returns[returns != 0]
    return (traded > 0).mean() if len(traded) else 0

def run_backtest(pred_prob, future_returns, threshold=0.55, transaction_cost=0.0005):
    direction = np.zeros(len(pred_prob))
    direction[pred_prob > threshold] = 1
    direction[pred_prob < (1 - threshold)] = -1

    returns = direction * future_returns
    trade_mask = direction != 0
    returns[trade_mask] -= transaction_cost
    cumulative = (1 + returns).cumprod()

    return {
        "returns": returns,
        "cumulative": cumulative,
        "sharpe": sharpe_ratio(returns),
        "max_drawdown": max_drawdown(cumulative),
        "win_rate": win_rate(returns),
        "total_return": cumulative[-1] - 1,
        "n_trades": trade_mask.sum()
    }



In [ ]:
test_future_returns = test_df["future_return"].values[-len(test_pred_prob):]
result = run_backtest(test_pred_prob, test_future_returns)
buyhold_cumulative = (1 + test_future_returns).cumprod()

print(f"Strategy  -> Sharpe: {result['sharpe']:.3f} | MaxDD: {result['max_drawdown']*100:.2f}% | "
      f"Return: {result['total_return']*100:.2f}% | WinRate: {result['win_rate']*100:.2f}% | Trades: {result['n_trades']}")
print(f"Buy&Hold  -> Sharpe: {sharpe_ratio(test_future_returns):.3f} | "
      f"MaxDD: {max_drawdown(buyhold_cumulative)*100:.2f}% | Return: {(buyhold_cumulative[-1]-1)*100:.2f}%")



In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(result["cumulative"], label="Model Strategy")
plt.plot(buyhold_cumulative, label="Buy & Hold")
plt.xlabel("Test period (hours)")
plt.ylabel("Cumulative growth")
plt.title("Strategy vs Buy & Hold")
plt.legend()
plt.show()

In [ ]:
N_SEGMENTS = 4
segment_size = len(result["returns"]) // N_SEGMENTS

print(f"{'Segment':<10}{'Sharpe':<10}{'Return %':<12}{'MaxDD %':<10}{'Trades':<8}")
for i in range(N_SEGMENTS):
    start = i * segment_size
    end = (i + 1) * segment_size if i < N_SEGMENTS - 1 else len(result["returns"])
    seg_returns = result["returns"][start:end]
    seg_cumulative = (1 + seg_returns).cumprod()
    print(f"{i+1:<10}{sharpe_ratio(seg_returns):<10.3f}{(seg_cumulative[-1]-1)*100:<12.2f}"
          f"{max_drawdown(seg_cumulative)*100:<10.2f}{(seg_returns!=0).sum():<8}")